# QUERY 5
Cantidad de transacciones del período [2022-09-01, 2022-09-05] con formato de pago
"Wire" o "ACH" cuyo monto convertido a USD sea menor a 1

In [1]:
import numpy as np
import pandas as pd
import requests

RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO = "transacciones_sample.csv"
SOLUCION = "q5_solucion.csv"

transacciones = pd.read_csv(f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO}")
print(f"Total filas cargadas: {len(transacciones)}")
transacciones.head()


Total filas cargadas: 1000


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/08 03:47,248957,8122D0BB0,248957,8124BDFE0,17338.88,Brazil Real,17338.88,Brazil Real,Credit Card,0
1,2022/09/09 16:36,20,8186978E0,26894,81BE09EC0,907.52,US Dollar,907.52,US Dollar,Cheque,0
2,2022/09/04 19:24,70,10042B858,136136,80E23F190,9104.75,Canadian Dollar,9104.75,Canadian Dollar,Cheque,0
3,2022/09/01 09:46,64657,817A6EFF0,166753,818A0D010,2543.96,Saudi Riyal,2543.96,Saudi Riyal,Cheque,0
4,2022/09/10 11:56,14,8072F5E80,14,8089BC8A0,4411.67,Rupee,4411.67,Rupee,Cheque,0


In [2]:
# Filtro por período [2022-09-01, 2022-09-05]
# "2022/09/01" <= Timestamp <= "2022/09/05" (inclusive)
INICIO = "2022/09/01"
FIN    = "2022/09/06"  # "2022/09/06" para incluir todo el día 5/9

filtro_periodo = transacciones[
    (transacciones["Timestamp"] >= INICIO) &
    (transacciones["Timestamp"] < FIN)
]
print(f"Transacciones en el período: {len(filtro_periodo)}")
filtro_periodo.head()

Transacciones en el período: 547


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
2,2022/09/04 19:24,70,10042B858,136136,80E23F190,9104.75,Canadian Dollar,9104.75,Canadian Dollar,Cheque,0
3,2022/09/01 09:46,64657,817A6EFF0,166753,818A0D010,2543.96,Saudi Riyal,2543.96,Saudi Riyal,Cheque,0
5,2022/09/03 16:22,64059,8178F0540,64057,8180635F0,1238.00,Saudi Riyal,1238.00,Saudi Riyal,Cheque,0
6,2022/09/05 18:33,70,10042B738,12,80828EFD0,304293.82,Yen,304293.82,Yen,Cheque,0
8,2022/09/01 14:19,3533,800885190,3533,800885190,5436.03,US Dollar,5436.03,US Dollar,Reinvestment,0


In [3]:
# Filtro por formato de pago: Wire o ACH
FORMATOS = ["Wire", "ACH"]

filtro_formato = filtro_periodo[filtro_periodo["Payment Format"].isin(FORMATOS)]
print(f"Transacciones Wire/ACH en el período: {len(filtro_formato)}")
filtro_formato.head()


Transacciones Wire/ACH en el período: 75


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
18,2022/09/01 05:18,21260,803DB58D0,21260,803DB58D0,824.26,Euro,965.86,US Dollar,ACH,0
28,2022/09/03 09:11,20,80303E270,25307,80303F3D0,384667.00,Euro,384667.00,Euro,ACH,0
52,2022/09/02 18:10,324326,80DC30EE0,22129,80DC31C20,355.40,US Dollar,355.40,US Dollar,ACH,0
61,2022/09/02 15:37,12274,80945F030,158363,81AB03E80,0.28,Euro,0.28,Euro,Wire,0
79,2022/09/04 20:19,116648,80DBBE530,238824,80F8B4370,23255.79,US Dollar,23255.79,US Dollar,ACH,0


In [4]:
# Descarga de cotizaciones desde API Frankfurter (base EUR)
# y forward-fill para días sin mercado (fines de semana / feriados)
from datetime import date, timedelta

url = "https://api.frankfurter.app/2022-09-01..2022-09-05"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
cotizaciones_raw = resp.json()["rates"]   # solo días hábiles
print("Fechas con cotización de la API:", list(cotizaciones_raw.keys()))

# Expandir el rango completo usando la última cotización conocida
cotizaciones = {}
last_rates = None
dia = date(2022, 9, 1)
fin = date(2022, 9, 5)
while dia <= fin:
    key = dia.strftime("%Y-%m-%d")
    if key in cotizaciones_raw:
        last_rates = cotizaciones_raw[key]
    if last_rates is not None:
        cotizaciones[key] = last_rates
    dia += timedelta(days=1)

print("Fechas disponibles tras forward-fill:", list(cotizaciones.keys()))


Fechas con cotización de la API: ['2022-09-01', '2022-09-02', '2022-09-05']
Fechas disponibles tras forward-fill: ['2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04', '2022-09-05']


In [5]:
# Conversión de monto recibido a USD (misma lógica que el worker distribuido)
CURRENCY_MAP = {
    "US Dollar":         "USD",
    "Euro":              "EUR",
    "UK Pound":          "GBP",
    "Yen":               "JPY",
    "Australian Dollar": "AUD",
    "Bitcoin":           "BTC",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Mexican Peso":      "MXN",
    "Ruble":             "RUB",
    "Rupee":             "INR",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "Yuan":              "CNY",
}

def convertir_a_usd(row):
    iso = CURRENCY_MAP.get(row["Receiving Currency"])
    if not iso:
        return None
    if iso == "USD":
        return float(row["Amount Received"])

    fecha = row["Timestamp"].split(" ")[0].replace("/", "-")
    rates = cotizaciones.get(fecha)  # ya tiene forward-fill para fines de semana
    if not rates:
        return None

    rate_usd = rates.get("USD")
    if not rate_usd:
        return None

    monto = float(row["Amount Received"])
    if iso == "EUR":
        return monto * rate_usd

    # Triangulación: origen → EUR → USD
    rate_origen = rates.get(iso)
    if not rate_origen:
        return None
    return (monto / rate_origen) * rate_usd

filtro_formato = filtro_formato.copy()
filtro_formato["amount_usd"] = filtro_formato.apply(convertir_a_usd, axis=1)

# Descartamos filas sin cotización (divisa no mapeada) y filtramos monto < 1 USD
menores_1_usd = filtro_formato.dropna(subset=["amount_usd"])
menores_1_usd = menores_1_usd[menores_1_usd["amount_usd"] < 1.0]
print(f"Transacciones < 1 USD: {len(menores_1_usd)}")
menores_1_usd[["Timestamp", "Receiving Currency", "Amount Received", "amount_usd"]].head()


Transacciones < 1 USD: 1


,Timestamp,Receiving Currency,Amount Received,amount_usd
61,2022/09/02 15:37,Euro,0.28,0.279804


In [6]:
# Resultado final: cantidad de transacciones
count = len(menores_1_usd)
resultado = pd.DataFrame({"count": [count]})
print(f"Q5 resultado: {count} transacciones")

resultado.to_csv(SOLUCION, index=False)
print(f"Guardado en {SOLUCION}")


Q5 resultado: 1 transacciones
Guardado en q5_solucion.csv
